In [1]:
from pathlib import Path

import h3
import polars as pl
import polars_h3 as plh3
import yaml


def update(data: pl.DataFrame, dissolved: list[str]) -> pl.DataFrame:
    return data.with_columns(
        source=pl.when(pl.col("source").is_in(dissolved))
        .then(pl.col("source"))
        .otherwise(plh3.cell_to_parent("source", 7)),
        target=pl.when(pl.col("target").is_in(dissolved))
        .then(pl.col("target"))
        .otherwise(plh3.cell_to_parent("target", 7)),
    )


def to_locations(hexagons: list[str]) -> pl.DataFrame:
    records = []
    for h in hexagons:
        lat, lon = h3.cell_to_latlng(h)
        records.append([h, lat, lon])
    return pl.from_records(records, schema=["id", "lat", "lon"], orient="row")

In [2]:
with open("../config.budapest.yaml") as fp:
    config = yaml.safe_load(fp)

In [72]:
RES = 9
DIR = f"{config['dirs']['output']}/{config['city']}"
AGGREGATION = "device"  # device | trip
STRICT = True

In [73]:
# mixed_grid = gpd.read_file("../output/grid/multi_grid_company_distribution.geojson")
# ids = pl.from_pandas(mixed_grid[["h3"]])

# to_locations(ids["h3"].to_list()).write_csv("../output/network_mixedgrid/locations.csv")

# cartesian_product = ids.join(ids, how="cross")
# cartesian_product.columns = ["source", "target"]
# cartesian_product = cartesian_product.with_columns(weight=pl.lit(0))


# budapest_h3_8 = gpd.read_file("../data/budapest_h3_8.geojson")
# in_budapest = budapest_h3_8["h3_polyfill"].tolist()

# with open("../output/grid/dissolved.json", "r") as fp:
#     dissolved = json.load(fp)["dissolved"]

In [74]:
st_at_hw = pl.read_csv(
    f"{DIR}/dynamic_commuting/{RES}/number_of_stops_at_home_work.csv"
)
st_at_hw.head(2)

device_id,day,stops_at_home,stops_at_work,h3_9_home,count_home,h3_9_work,count_work
i64,str,i64,i64,str,i64,str,i64
1,"""2025-05-04""",1,0,"""891e037850fffff""",28,"""891e037850fffff""",37
1,"""2025-05-05""",1,5,"""891e037850fffff""",28,"""891e037850fffff""",37


In [75]:
dyn_comm = st_at_hw.filter(
    (pl.col.count_home >= 5) & (pl.col.count_work >= 5) & (pl.col.stops_at_work > 0)
)
if STRICT:
    dyn_comm = dyn_comm.filter(pl.col.stops_at_home > 0)
if AGGREGATION == "trip":
    dyn_comm = dyn_comm.group_by(["day", f"h3_{RES}_home", f"h3_{RES}_work"]).agg(
        pl.col.device_id.count().alias("count")
    )
elif AGGREGATION == "device":
    dyn_comm = dyn_comm.group_by(["day", f"h3_{RES}_home", f"h3_{RES}_work"]).agg(
        pl.col.device_id.unique().count().alias("count")
    )

In [76]:
evaluation = "strict"
if STRICT:
    evaluation = "relaxed"


In [77]:
Path(f"{DIR}/dynamic_commuting/{RES}/{AGGREGATION}/{evaluation}").mkdir(
    parents=True, exist_ok=True
)
dyn_comm.sort(["day", f"h3_{RES}_home", f"h3_{RES}_work"]).write_csv(
    f"{DIR}/dynamic_commuting/{RES}/{AGGREGATION}/{evaluation}/daily_commuting.csv"
)

In [78]:
dyn_comm.to_pandas()["count"].describe()

count    2.346134e+06
mean     2.110175e+00
std      1.919106e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      3.100000e+01
Name: count, dtype: float64